# Analyze AIT Log v2 Attack Phases

In [114]:
import pandas as pd

from utils import analyze_phase, analyze_origin_destination

## Load MSA Data

In [115]:
dataset = "aitv2"
scenario = "santos"
data_dir = f"../data/interim/{dataset}/{scenario}/flows_labeled"
file_path = f"{data_dir}/flows_augmented.csv"

In [116]:
df = pd.read_csv(file_path)

In [117]:
print("Phase distribution:")
df.value_counts("phase")

Phase distribution:


phase
0    333964
1     12861
2      5670
4       228
3        26
Name: count, dtype: int64

## Feature Analysis

In [118]:
categorical_cols = [
    "sport",
    "dport",
    "proto",
    "service",
    "conn_state", 
    "local_orig",
    "local_resp",
    "history",    
    "tunnel_parents",
    "ip_proto",   
]

In [119]:
for col in categorical_cols:
    df_col = df[col]
    df_col_count = df_col.value_counts()
    print(f"Unique values in {col}: {len(df_col_count)}")
    # print(df_col_count)

Unique values in sport: 47841
Unique values in dport: 1084
Unique values in proto: 4
Unique values in service: 15
Unique values in conn_state: 12
Unique values in local_orig: 1
Unique values in local_resp: 2
Unique values in history: 1254
Unique values in tunnel_parents: 1
Unique values in ip_proto: 5


## Per Phase Analysis

In [120]:
df_benign = df[df["phase"] == 0]
df_phase_1 = df[df["phase"] == 1]
df_phase_2 = df[df["phase"] == 2]
df_phase_3 = df[df["phase"] == 3]
df_phase_4 = df[df["phase"] == 4]   

### Phase 1 - Scanning

In [121]:
phase_1_labels = df_phase_1["label"].value_counts()
print("Phase 1 Label Distribution:")
print(phase_1_labels)

Phase 1 Label Distribution:
label
data exfiltration    12861
Name: count, dtype: int64


In [122]:
analyze_phase(df_phase_2, "Phase 2")

Total Flows: 5670

 --- IP distribution ---

Source IPs (2):
src_ip
172.21.128.119    5074
10.229.2.216       596
Name: count, dtype: int64

Destination IPs (263):
dst_ip
192.168.104.155    1035
172.21.131.50       896
172.21.128.54       870
192.168.104.218     841
10.229.255.254      596
                   ... 
192.168.104.95        1
192.168.104.198       1
192.168.104.141       1
192.168.104.81        1
192.168.104.98        1
Name: count, Length: 263, dtype: int64

 --- Port distribution ---
Source Ports (3807):
sport
41554    6
60056    6
49810    5
56148    5
44526    5
        ..
47190    1
54112    1
53852    1
41092    1
34884    1
Name: count, Length: 3807, dtype: int64

Destination Ports (592):
dport
443     834
80      589
587      26
993      14
139      14
       ... 
146       5
8873      5
5633      5
800       5
1088      5
Name: count, Length: 592, dtype: int64

 --- Protocol distribution ---
proto
tcp    5670
Name: count, dtype: int64

 --- Service distribution ---


(src_ip
 172.21.128.119    5074
 10.229.2.216       596
 Name: count, dtype: int64,
 dst_ip
 192.168.104.155    1035
 172.21.131.50       896
 172.21.128.54       870
 192.168.104.218     841
 10.229.255.254      596
                    ... 
 192.168.104.95        1
 192.168.104.198       1
 192.168.104.141       1
 192.168.104.81        1
 192.168.104.98        1
 Name: count, Length: 263, dtype: int64,
 sport
 41554    6
 60056    6
 49810    5
 56148    5
 44526    5
         ..
 47190    1
 54112    1
 53852    1
 41092    1
 34884    1
 Name: count, Length: 3807, dtype: int64,
 dport
 443     834
 80      589
 587      26
 993      14
 139      14
        ... 
 146       5
 8873      5
 5633      5
 800       5
 1088      5
 Name: count, Length: 592, dtype: int64)

In [123]:
phase_1_labels = df_phase_1["label"].unique()

analyze_phase(df_phase_1, "Phase 1")

# for label in phase_1_labels:
#     df_label = df_phase_1[df_phase_1["label"] == label]
#     print(f"\nLabel: {label}")
#     analyze_phase(df_label, label)

Total Flows: 12861

 --- IP distribution ---

Source IPs (1):
src_ip
10.229.255.254    12861
Name: count, dtype: int64

Destination IPs (1):
dst_ip
10.229.2.216    12861
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (11598):
sport
1772     4
8931     4
6966     4
60954    3
62882    3
        ..
58934    1
18514    1
48402    1
25045    1
23702    1
Name: count, Length: 11598, dtype: int64

Destination Ports (1):
dport
53    12861
Name: count, dtype: int64

 --- Protocol distribution ---
proto
udp    12861
Name: count, dtype: int64

 --- Service distribution ---
service
dns    12861
Name: count, dtype: int64


(src_ip
 10.229.255.254    12861
 Name: count, dtype: int64,
 dst_ip
 10.229.2.216    12861
 Name: count, dtype: int64,
 sport
 1772     4
 8931     4
 6966     4
 60954    3
 62882    3
         ..
 58934    1
 18514    1
 48402    1
 25045    1
 23702    1
 Name: count, Length: 11598, dtype: int64,
 dport
 53    12861
 Name: count, dtype: int64)

In [124]:
augmented_features = [
    "unique_sources",  
    "fanin_rate",
    "unique_targets",
    "fanout_rate",
    "dst_ratio",
    "unique_ports",
    "connection_count",
]

In [125]:
df_augmented_stats = df_phase_1[augmented_features].describe()
print("\nPhase 1 Statistics:")
print(df_augmented_stats)


Phase 1 Statistics:
       unique_sources    fanin_rate  unique_targets   fanout_rate  dst_ratio  \
count         12861.0  12861.000000         12861.0  12861.000000    12861.0   
mean              1.0      0.070983             1.0      0.070983        1.0   
std               0.0      0.007569             0.0      0.007569        0.0   
min               1.0      0.016667             1.0      0.016667        1.0   
25%               1.0      0.066667             1.0      0.066667        1.0   
50%               1.0      0.066667             1.0      0.066667        1.0   
75%               1.0      0.083333             1.0      0.083333        1.0   
max               1.0      0.116667             1.0      0.116667        1.0   

       unique_ports  connection_count  
count       12861.0      12861.000000  
mean            1.0       6436.091828  
std             0.0       3714.982223  
min             1.0          1.000000  
25%             1.0       3219.000000  
50%             1.

In [126]:
df_augmented_stats = df_phase_2[augmented_features].describe()
print("\nPhase 2 Statistics:")
print(df_augmented_stats)


Phase 2 Statistics:
       unique_sources   fanin_rate  unique_targets  fanout_rate    dst_ratio  \
count     5670.000000  5670.000000     5670.000000  5670.000000  5670.000000   
mean         2.387302     9.100479      203.681834    38.118236     0.102003   
std          0.692358     7.940129      105.055718    25.681492     0.164301   
min          2.000000     0.033333        1.000000     0.016667     0.000038   
25%          2.000000     1.620833      251.000000    13.220833     0.005055   
50%          2.000000     7.416667      262.000000    37.325000     0.020714   
75%          3.000000    15.112500      262.000000    60.562500     0.116756   
max          7.000000    27.383333      262.000000    83.516667     0.670807   

       unique_ports  connection_count  
count   5670.000000       5670.000000  
mean     385.587478      25331.665256  
std      138.298767       8569.061934  
min        8.000000        371.000000  
25%      314.000000      26495.250000  
50%      366.00000

In [127]:
df_augmented_stats = df_phase_3[augmented_features].describe()
print("\nPhase 3 Statistics:")
print(df_augmented_stats)


Phase 3 Statistics:
       unique_sources  fanin_rate  unique_targets  fanout_rate  dst_ratio  \
count            26.0   26.000000            26.0    26.000000  26.000000   
mean              2.0    1.835256             3.0     1.277564   0.126576   
std               0.0    1.183989             0.0     0.771707   0.000507   
min               2.0    0.983333             3.0     0.700000   0.125637   
25%               2.0    1.133333             3.0     0.833333   0.126313   
50%               2.0    1.200000             3.0     0.866667   0.126717   
75%               2.0    2.279167             3.0     1.545833   0.126943   
max               2.0    4.433333             3.0     2.983333   0.127233   

       unique_ports  connection_count  
count          26.0         26.000000  
mean          595.0      30823.807692  
std             0.0         20.071909  
min           595.0      30787.000000  
25%           595.0      30814.250000  
50%           595.0      30828.500000  
75%  

In [128]:
df_augmented_stats = df_phase_4[augmented_features].describe()
print("\nPhase 1 Statistics:")
print(df_augmented_stats)


Phase 1 Statistics:
       unique_sources  fanin_rate  unique_targets  fanout_rate   dst_ratio  \
count      228.000000  228.000000      228.000000   228.000000  228.000000   
mean         4.425439    0.435599        2.622807     0.334064    0.330142   
std          1.970859    0.274488        1.934570     0.222173    0.111354   
min          1.000000    0.016667        1.000000     0.016667    0.000015   
25%          3.000000    0.212500        1.000000     0.116667    0.384122   
50%          5.000000    0.375000        2.000000     0.333333    0.385252   
75%          6.000000    0.650000        3.000000     0.550000    0.385690   
max          8.000000    1.200000       12.000000     0.883333    0.386206   

       unique_ports  connection_count  
count    228.000000        228.000000  
mean     580.039474      31044.828947  
std       91.477366       5304.626207  
min        4.000000        264.000000  
25%      595.000000      30904.750000  
50%      595.000000      30962.50000

In [129]:
analyze_origin_destination(df_phase_1, "Phase 1")

Phase 1 Origin Distribution:
local_orig
T    12861
Name: count, dtype: int64
Phase 1 Destination Distribution:
local_resp
T    12861
Name: count, dtype: int64


In [130]:
# df_phase_1.to_csv(f"{data_dir}/phase_1_flows.csv", index=False)

In [131]:
def analyze_aug_features(df, feature, phase_name):
    print(f"\n{feature} Statistics for {phase_name}:")
    feature_low = df[df[feature] <= 0.2]
    feature_medium = df[(df[feature] > 0.2) & (df[feature] <= 0.5)]
    feature_high = df[df[feature] > 0.5]
    print(f"\n{feature} <= 0.2:")
    print(feature_low[feature].describe())
    print(f"\n{feature} > 0.2 and <= 0.5:")
    print(feature_medium[feature].describe())
    print(f"\n{feature} > 0.5:")
    print(feature_high[feature].describe())

In [132]:
def analyze_aug_features(df, feature, phase_name):
    print(f"\n{feature} Statistics for {phase_name}:")
    feature_low = df[df[feature] <= 5]
    feature_medium = df[(df[feature] > 5) & (df[feature] <= 10)]
    feature_high = df[df[feature] > 10]
    print(f"\n{feature} <= 5:")
    print(feature_low[feature].describe())
    print(f"\n{feature} > 5 and <= 10:")
    print(feature_medium[feature].describe())
    print(f"\n{feature} > 10:")
    print(feature_high[feature].describe())

In [133]:
analyze_aug_features(df_phase_1, "unique_targets", "Phase 1")


unique_targets Statistics for Phase 1:

unique_targets <= 5:
count    12861.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: unique_targets, dtype: float64

unique_targets > 5 and <= 10:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: unique_targets, dtype: float64

unique_targets > 10:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: unique_targets, dtype: float64


### Phase 2 - Exploitation

In [134]:
print("Number of samples in Phase 2:", len(df_phase_2))

Number of samples in Phase 2: 5670


In [135]:
analyze_origin_destination(df_phase_2, "Phase 2")

Phase 2 Origin Distribution:
local_orig
T    5670
Name: count, dtype: int64
Phase 2 Destination Distribution:
local_resp
T    5670
Name: count, dtype: int64


In [136]:
analyze_phase(df_phase_2, "Phase 2")

Total Flows: 5670

 --- IP distribution ---

Source IPs (2):
src_ip
172.21.128.119    5074
10.229.2.216       596
Name: count, dtype: int64

Destination IPs (263):
dst_ip
192.168.104.155    1035
172.21.131.50       896
172.21.128.54       870
192.168.104.218     841
10.229.255.254      596
                   ... 
192.168.104.95        1
192.168.104.198       1
192.168.104.141       1
192.168.104.81        1
192.168.104.98        1
Name: count, Length: 263, dtype: int64

 --- Port distribution ---
Source Ports (3807):
sport
41554    6
60056    6
49810    5
56148    5
44526    5
        ..
47190    1
54112    1
53852    1
41092    1
34884    1
Name: count, Length: 3807, dtype: int64

Destination Ports (592):
dport
443     834
80      589
587      26
993      14
139      14
       ... 
146       5
8873      5
5633      5
800       5
1088      5
Name: count, Length: 592, dtype: int64

 --- Protocol distribution ---
proto
tcp    5670
Name: count, dtype: int64

 --- Service distribution ---


(src_ip
 172.21.128.119    5074
 10.229.2.216       596
 Name: count, dtype: int64,
 dst_ip
 192.168.104.155    1035
 172.21.131.50       896
 172.21.128.54       870
 192.168.104.218     841
 10.229.255.254      596
                    ... 
 192.168.104.95        1
 192.168.104.198       1
 192.168.104.141       1
 192.168.104.81        1
 192.168.104.98        1
 Name: count, Length: 263, dtype: int64,
 sport
 41554    6
 60056    6
 49810    5
 56148    5
 44526    5
         ..
 47190    1
 54112    1
 53852    1
 41092    1
 34884    1
 Name: count, Length: 3807, dtype: int64,
 dport
 443     834
 80      589
 587      26
 993      14
 139      14
        ... 
 146       5
 8873      5
 5633      5
 800       5
 1088      5
 Name: count, Length: 592, dtype: int64)

In [137]:
df_phase_2.columns

Index(['flow_id', 'start_time', 'end_time', 'duration', 'src_ip', 'sport',
       'dst_ip', 'dport', 'proto', 'service', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents', 'ip_proto', 'label', 'sensor_host', 'start_time_dt',
       'end_time_dt', 'phase', 'unique_sources', 'fanin_rate',
       'unique_targets', 'fanout_rate', 'dst_ratio', 'unique_ports',
       'connection_count'],
      dtype='object')

In [138]:
phase_2_labels = df_phase_2["label"].value_counts()
print("Phase 2 Label Distribution:")
print(phase_2_labels)

Phase 2 Label Distribution:
label
service_scan             4056
host_discover_local      1000
dns_brute_force_start     325
host_discover_dmz         115
dirb_scan                  90
wpscan                     84
Name: count, dtype: int64


In [139]:
phase_2_labels = df_phase_2["label"].unique()
print("Unique labels in Phase 2:", phase_2_labels)

Unique labels in Phase 2: ['host_discover_dmz' 'dns_brute_force_start' 'host_discover_local'
 'service_scan' 'dirb_scan' 'wpscan']


In [140]:
phase_2_protocols = df_phase_2["proto"].value_counts()
print("Phase 2 Protocol Distribution:")
print(phase_2_protocols)

Phase 2 Protocol Distribution:
proto
tcp    5670
Name: count, dtype: int64


In [141]:
phase_2_ports = df_phase_2["dport"].value_counts()
print("Phase 2 Destination Port Distribution:")
print(phase_2_ports)

Phase 2 Destination Port Distribution:
dport
443     834
80      589
587      26
993      14
139      14
       ... 
146       5
8873      5
5633      5
800       5
1088      5
Name: count, Length: 592, dtype: int64


In [142]:
# Label = host discover local
curr_label = phase_2_labels[0]
print(f"Analyzing Phase 2 - {curr_label}")
df_host_discover_local = df_phase_2[df_phase_2["label"] == curr_label]
analyze_origin_destination(df_host_discover_local, f"Phase 2 - {curr_label}")
analyze_phase(df_host_discover_local, f"Phase 2 - {curr_label}")

Analyzing Phase 2 - host_discover_dmz
Phase 2 - host_discover_dmz Origin Distribution:
local_orig
T    115
Name: count, dtype: int64
Phase 2 - host_discover_dmz Destination Distribution:
local_resp
T    115
Name: count, dtype: int64
Total Flows: 115

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    115
Name: count, dtype: int64

Destination IPs (6):
dst_ip
172.21.129.224    98
172.21.131.122     6
172.21.128.54      4
172.21.128.2       4
172.21.131.50      2
172.21.128.1       1
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (60):
sport
35596    4
39150    3
34464    2
37488    2
48880    2
36224    2
48882    2
35104    2
58794    2
32924    2
54296    2
56452    2
35554    2
35556    2
48884    2
48676    2
35558    2
35560    2
35552    2
35550    2
48890    2
48888    2
35570    2
35568    2
35566    2
48886    2
35564    2
35562    2
35582    2
48898    2
35584    2
35586    2
35588    2
35590    2
35592    2
35594    2
48892    2
48894    2


(src_ip
 172.21.128.119    115
 Name: count, dtype: int64,
 dst_ip
 172.21.129.224    98
 172.21.131.122     6
 172.21.128.54      4
 172.21.128.2       4
 172.21.131.50      2
 172.21.128.1       1
 Name: count, dtype: int64,
 sport
 35596    4
 39150    3
 34464    2
 37488    2
 48880    2
 36224    2
 48882    2
 35104    2
 58794    2
 32924    2
 54296    2
 56452    2
 35554    2
 35556    2
 48884    2
 48676    2
 35558    2
 35560    2
 35552    2
 35550    2
 48890    2
 48888    2
 35570    2
 35568    2
 35566    2
 48886    2
 35564    2
 35562    2
 35582    2
 48898    2
 35584    2
 35586    2
 35588    2
 35590    2
 35592    2
 35594    2
 48892    2
 48894    2
 48896    2
 35572    2
 35574    2
 35576    2
 35578    2
 35580    2
 35598    2
 35600    2
 35602    2
 35604    2
 35606    2
 35608    2
 35610    2
 35612    2
 44198    1
 46436    1
 39822    1
 58320    1
 37884    1
 35922    1
 39992    1
 42112    1
 Name: count, dtype: int64,
 dport
 443    94


In [143]:
# Label = host discover local
df_host_discover_local = df_phase_2[df_phase_2["label"] == "host_discover_local"]
# analyze_origin_destination(df_host_discover_local, "Phase 2 - Host Discover Local")
analyze_phase(df_host_discover_local, "Phase 2 - Host Discover Local")

Total Flows: 1000

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    1000
Name: count, dtype: int64

Destination IPs (256):
dst_ip
192.168.104.205    12
192.168.104.3       4
192.168.104.6       4
192.168.104.7       4
192.168.104.8       4
                   ..
192.168.104.81      1
192.168.104.98      1
192.168.104.198     1
192.168.104.141     1
192.168.104.4       1
Name: count, Length: 256, dtype: int64

 --- Port distribution ---
Source Ports (950):
sport
42854    3
42672    3
46724    3
59342    3
40434    3
        ..
40794    1
47938    1
46144    1
35360    1
58640    1
Name: count, Length: 950, dtype: int64

Destination Ports (2):
dport
80     512
443    488
Name: count, dtype: int64

 --- Protocol distribution ---
proto
tcp    1000
Name: count, dtype: int64

 --- Service distribution ---
service
-    1000
Name: count, dtype: int64


(src_ip
 172.21.128.119    1000
 Name: count, dtype: int64,
 dst_ip
 192.168.104.205    12
 192.168.104.3       4
 192.168.104.6       4
 192.168.104.7       4
 192.168.104.8       4
                    ..
 192.168.104.81      1
 192.168.104.98      1
 192.168.104.198     1
 192.168.104.141     1
 192.168.104.4       1
 Name: count, Length: 256, dtype: int64,
 sport
 42854    3
 42672    3
 46724    3
 59342    3
 40434    3
         ..
 40794    1
 47938    1
 46144    1
 35360    1
 58640    1
 Name: count, Length: 950, dtype: int64,
 dport
 80     512
 443    488
 Name: count, dtype: int64)

In [144]:
def attack_frequency(df, phase_name):
    time_range = df["end_time"].max() - df["start_time"].min()
    attack_count = len(df)
    frequency = attack_count / time_range if time_range > 0 else 0
    print(f"{phase_name} Attack Frequency: {frequency:.4f} attacks per second")

In [145]:
print(f"Analyzing Phase 2:\n")
for curr_label in phase_2_labels:
    print(f"=== {curr_label} ===")
    df_curr_label = df_phase_2[df_phase_2["label"] == curr_label]
    # analyze_origin_destination(df_curr_label, f"Phase 2 - {curr_label}")
    analyze_phase(df_curr_label, f"Phase 2 - {curr_label}")
    # attack_frequency(df_curr_label, f"Phase 2 - {curr_label}")
    print()

Analyzing Phase 2:

=== host_discover_dmz ===
Total Flows: 115

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    115
Name: count, dtype: int64

Destination IPs (6):
dst_ip
172.21.129.224    98
172.21.131.122     6
172.21.128.54      4
172.21.128.2       4
172.21.131.50      2
172.21.128.1       1
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (60):
sport
35596    4
39150    3
34464    2
37488    2
48880    2
36224    2
48882    2
35104    2
58794    2
32924    2
54296    2
56452    2
35554    2
35556    2
48884    2
48676    2
35558    2
35560    2
35552    2
35550    2
48890    2
48888    2
35570    2
35568    2
35566    2
48886    2
35564    2
35562    2
35582    2
48898    2
35584    2
35586    2
35588    2
35590    2
35592    2
35594    2
48892    2
48894    2
48896    2
35572    2
35574    2
35576    2
35578    2
35580    2
35598    2
35600    2
35602    2
35604    2
35606    2
35608    2
35610    2
35612    2
44198    1
46436    1
39822    1


### Phase 3 Password Cracking

In [146]:
df_phase_3.head(10)

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,start_time_dt,end_time_dt,phase,unique_sources,fanin_rate,unique_targets,fanout_rate,dst_ratio,unique_ports,connection_count
311919,f24386,1.642419e+09,1.642419e+09,3.172493,172.21.128.119,44674,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:22:57.162303925,2022-01-17 11:23:00.334796906,3,2,4.416667,3,2.966667,0.125637,595,30787
311920,f12338,1.642419e+09,1.642419e+09,3.171754,172.21.128.119,44674,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:22:57.163022995,2022-01-17 11:23:00.334776878,3,2,4.433333,3,2.983333,0.125666,595,30788
311927,f24388,1.642419e+09,1.642419e+09,0.095648,172.21.128.119,44678,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:03.510241984,2022-01-17 11:23:03.605890036,3,2,4.266667,3,2.866667,0.125751,595,30791
311928,f12340,1.642419e+09,1.642419e+09,0.094994,172.21.128.119,44678,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:03.510807990,2022-01-17 11:23:03.605802059,3,2,4.283333,3,2.883333,0.125779,595,30792
311982,f24392,1.642419e+09,1.642419e+09,0.093564,172.21.128.119,44686,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:16.512259959,2022-01-17 11:23:16.605823994,3,2,2.466667,3,1.666667,0.125978,595,30799
311983,f12344,1.642419e+09,1.642419e+09,0.092939,172.21.128.119,44686,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:16.512897014,2022-01-17 11:23:16.605835915,3,2,2.483333,3,1.683333,0.126006,595,30800
312016,f24400,1.642419e+09,1.642419e+09,0.150203,172.21.128.119,44698,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:31.353312969,2022-01-17 11:23:31.503515959,3,2,2.266667,3,1.533333,0.126306,595,30814
312017,f12351,1.642419e+09,1.642419e+09,0.149455,172.21.128.119,44698,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:31.354079008,2022-01-17 11:23:31.503534079,3,2,2.283333,3,1.550000,0.126335,595,30815
312028,f24403,1.642419e+09,1.642419e+09,0.098250,172.21.128.119,44704,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:39.862638950,2022-01-17 11:23:39.960888863,3,2,1.400000,3,0.966667,0.126476,595,30820
312029,f12354,1.642419e+09,1.642419e+09,0.097714,172.21.128.119,44704,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:39.863159895,2022-01-17 11:23:39.960873842,3,2,1.416667,3,0.983333,0.126505,595,30821


In [147]:
df_phase_2_services = df_phase_2["service"].value_counts()
print("Phase 2 Service Distribution:")
print(df_phase_2_services)

Phase 2 Service Distribution:
service
-       5301
ssl      326
http      40
imap       2
dns        1
Name: count, dtype: int64


In [148]:
phase_3_services = df_phase_3["service"].value_counts()
print("Phase 3 Service Distribution:")
print(phase_3_services)

Phase 3 Service Distribution:
service
ssl    26
Name: count, dtype: int64


In [149]:
phase_3_labels = df_phase_3["label"].unique()
print("Unique labels in Phase 3:", phase_3_labels)

Unique labels in Phase 3: ['upload_rce_shell' 'check_user_id' 'check_date' 'check_network_config'
 'check_ps_a' 'read_resolv' 'read_passwd' 'check_release'
 'check_netstat_t' 'list_web_dir' 'read_group' 'check_wp_config'
 'dump_wp_users']


In [150]:
df_phase_3.head(50)

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,start_time_dt,end_time_dt,phase,unique_sources,fanin_rate,unique_targets,fanout_rate,dst_ratio,unique_ports,connection_count
311919,f24386,1.642419e+09,1.642419e+09,3.172493,172.21.128.119,44674,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:22:57.162303925,2022-01-17 11:23:00.334796906,3,2,4.416667,3,2.966667,0.125637,595,30787
311920,f12338,1.642419e+09,1.642419e+09,3.171754,172.21.128.119,44674,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:22:57.163022995,2022-01-17 11:23:00.334776878,3,2,4.433333,3,2.983333,0.125666,595,30788
311927,f24388,1.642419e+09,1.642419e+09,0.095648,172.21.128.119,44678,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:03.510241984,2022-01-17 11:23:03.605890036,3,2,4.266667,3,2.866667,0.125751,595,30791
311928,f12340,1.642419e+09,1.642419e+09,0.094994,172.21.128.119,44678,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:03.510807990,2022-01-17 11:23:03.605802059,3,2,4.283333,3,2.883333,0.125779,595,30792
311982,f24392,1.642419e+09,1.642419e+09,0.093564,172.21.128.119,44686,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:16.512259959,2022-01-17 11:23:16.605823994,3,2,2.466667,3,1.666667,0.125978,595,30799
311983,f12344,1.642419e+09,1.642419e+09,0.092939,172.21.128.119,44686,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:16.512897014,2022-01-17 11:23:16.605835915,3,2,2.483333,3,1.683333,0.126006,595,30800
312016,f24400,1.642419e+09,1.642419e+09,0.150203,172.21.128.119,44698,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:31.353312969,2022-01-17 11:23:31.503515959,3,2,2.266667,3,1.533333,0.126306,595,30814
312017,f12351,1.642419e+09,1.642419e+09,0.149455,172.21.128.119,44698,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:31.354079008,2022-01-17 11:23:31.503534079,3,2,2.283333,3,1.550000,0.126335,595,30815
312028,f24403,1.642419e+09,1.642419e+09,0.098250,172.21.128.119,44704,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:39.862638950,2022-01-17 11:23:39.960888863,3,2,1.400000,3,0.966667,0.126476,595,30820
312029,f12354,1.642419e+09,1.642419e+09,0.097714,172.21.128.119,44704,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:23:39.863159895,2022-01-17 11:23:39.960873842,3,2,1.416667,3,0.983333,0.126505,595,30821


In [151]:
mal_ip = "172.21.128.119"
df_non_malicious = df_phase_3[df_phase_3["src_ip"] != mal_ip]
df_non_malicious.head(10)

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,start_time_dt,end_time_dt,phase,unique_sources,fanin_rate,unique_targets,fanout_rate,dst_ratio,unique_ports,connection_count


In [152]:
for curr_label in phase_3_labels:
    print(f"Analyzing Phase 3 - {curr_label}")
    df_curr_label = df_phase_3[df_phase_3["label"] == curr_label]
    # analyze_origin_destination(df_curr_label, f"Phase 3 - {curr_label}")
    analyze_phase(df_curr_label, f"Phase 3 - {curr_label}")
    attack_frequency(df_curr_label, f"Phase 3 - {curr_label}")
    print(len(df_curr_label))
    print()

Analyzing Phase 3 - upload_rce_shell
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    2
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.104.155    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1):
sport
44674    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    2
Name: count, dtype: int64

 --- Protocol distribution ---
proto
tcp    2
Name: count, dtype: int64

 --- Service distribution ---
service
ssl    2
Name: count, dtype: int64
Phase 3 - upload_rce_shell Attack Frequency: 0.6304 attacks per second
2

Analyzing Phase 3 - check_user_id
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    2
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.104.155    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1):
sport
44678    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    2
Name: count, dtype: int64

 --- Protocol distributio

In [153]:
analyze_origin_destination(df_phase_3, "Phase 3")

Phase 3 Origin Distribution:
local_orig
T    26
Name: count, dtype: int64
Phase 3 Destination Distribution:
local_resp
T    26
Name: count, dtype: int64


In [154]:
analyze_phase(df_phase_3, "Phase 3")

Total Flows: 26

 --- IP distribution ---

Source IPs (1):
src_ip
172.21.128.119    26
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.104.155    26
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (13):
sport
44674    2
44678    2
44686    2
44698    2
44704    2
44706    2
44712    2
44714    2
44716    2
44720    2
44726    2
44728    2
44730    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    26
Name: count, dtype: int64

 --- Protocol distribution ---
proto
tcp    26
Name: count, dtype: int64

 --- Service distribution ---
service
ssl    26
Name: count, dtype: int64


(src_ip
 172.21.128.119    26
 Name: count, dtype: int64,
 dst_ip
 192.168.104.155    26
 Name: count, dtype: int64,
 sport
 44674    2
 44678    2
 44686    2
 44698    2
 44704    2
 44706    2
 44712    2
 44714    2
 44716    2
 44720    2
 44726    2
 44728    2
 44730    2
 Name: count, dtype: int64,
 dport
 443    26
 Name: count, dtype: int64)

In [155]:
analyze_origin_destination(df_phase_3, "Phase 3")

Phase 3 Origin Distribution:
local_orig
T    26
Name: count, dtype: int64
Phase 3 Destination Distribution:
local_resp
T    26
Name: count, dtype: int64


### Phase 4 - Data Exfiltration

In [156]:
analyze_phase(df_phase_4, "Phase 4")

Total Flows: 228

 --- IP distribution ---

Source IPs (4):
src_ip
172.21.128.119     222
10.229.0.4           3
10.229.1.118         2
192.168.104.155      1
Name: count, dtype: int64

Destination IPs (4):
dst_ip
172.21.129.224     182
192.168.104.155     40
10.229.2.216         3
10.229.2.25          3
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (114):
sport
44636    3
44732    2
44734    2
48904    2
44736    2
        ..
35800    2
48992    2
48990    2
48994    2
51280    1
Name: count, Length: 114, dtype: int64

Destination Ports (3):
dport
443    208
80      19
25       1
Name: count, dtype: int64

 --- Protocol distribution ---
proto
tcp    228
Name: count, dtype: int64

 --- Service distribution ---
service
ssl     208
http     19
smtp      1
Name: count, dtype: int64


(src_ip
 172.21.128.119     222
 10.229.0.4           3
 10.229.1.118         2
 192.168.104.155      1
 Name: count, dtype: int64,
 dst_ip
 172.21.129.224     182
 192.168.104.155     40
 10.229.2.216         3
 10.229.2.25          3
 Name: count, dtype: int64,
 sport
 44636    3
 44732    2
 44734    2
 48904    2
 44736    2
         ..
 35800    2
 48992    2
 48990    2
 48994    2
 51280    1
 Name: count, Length: 114, dtype: int64,
 dport
 443    208
 80      19
 25       1
 Name: count, dtype: int64)

In [157]:
df_phase_4_protocols = df_phase_4["dport"].value_counts()
print("Phase 4 Service Distribution:")
print(df_phase_4_protocols)

Phase 4 Service Distribution:
dport
443    208
80      19
25       1
Name: count, dtype: int64


In [158]:
df_phase_4_protocols = df_phase_4["proto"].value_counts()
print("Phase 4 Protocol Distribution:")
print(df_phase_4_protocols)

Phase 4 Protocol Distribution:
proto
tcp    228
Name: count, dtype: int64


In [159]:
df_phase_4.head(20)

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,start_time_dt,end_time_dt,phase,unique_sources,fanin_rate,unique_targets,fanout_rate,dst_ratio,unique_ports,connection_count
312125,f24418,1.642419e+09,1.642419e+09,16.408194,172.21.128.119,44732,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:24:16.209572077,2022-01-17 11:24:32.617765903,4,2,1.183333,3,0.866667,0.127261,595,30850
312126,f12377,1.642419e+09,1.642419e+09,16.407598,172.21.128.119,44732,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:24:16.210123062,2022-01-17 11:24:32.617721081,4,2,1.200000,3,0.883333,0.127289,595,30851
312142,f24420,1.642419e+09,1.642419e+09,1.101084,172.21.128.119,44734,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:24:35.566312075,2022-01-17 11:24:36.667396069,4,2,0.833333,2,0.583333,0.127318,595,30852
312143,f12379,1.642419e+09,1.642419e+09,1.100379,172.21.128.119,44734,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:24:35.567050934,2022-01-17 11:24:36.667429924,4,2,0.850000,2,0.600000,0.127346,595,30853
312145,f24423,1.642419e+09,1.642419e+09,5.261917,172.21.128.119,48904,172.21.129.224,443,tcp,ssl,...,2022-01-17 11:24:38.051532984,2022-01-17 11:24:43.313450098,4,5,0.250000,3,0.600000,0.383957,595,30855
312146,f86202,1.642419e+09,1.642419e+09,5.260738,172.21.128.119,48904,172.21.129.224,443,tcp,ssl,...,2022-01-17 11:24:38.052716017,2022-01-17 11:24:43.313453913,4,5,0.266667,3,0.616667,0.383977,595,30856
312151,f24421,1.642419e+09,1.642419e+09,2.123413,172.21.128.119,44736,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:24:39.177097082,2022-01-17 11:24:41.300510168,4,2,0.833333,3,0.633333,0.127362,595,30857
312152,f12381,1.642419e+09,1.642419e+09,2.123058,172.21.128.119,44736,192.168.104.155,443,tcp,ssl,...,2022-01-17 11:24:39.177728891,2022-01-17 11:24:41.300786972,4,2,0.850000,3,0.650000,0.127390,595,30858
312153,f12380,1.642419e+09,1.642419e+09,1.807480,192.168.104.155,44636,10.229.2.216,80,tcp,http,...,2022-01-17 11:24:39.469540119,2022-01-17 11:24:41.277020216,4,1,0.016667,4,0.166667,0.001119,124,894
312154,f78654,1.642419e+09,1.642419e+09,1.806820,10.229.0.4,44636,10.229.2.216,80,tcp,http,...,2022-01-17 11:24:39.470208883,2022-01-17 11:24:41.277028799,4,2,0.033333,6,0.150000,0.000015,9,65361


In [160]:
df_dst_ratio = df_phase_4["dst_ratio"].describe()
print("Phase 4 dst_ratio Statistics:")
print(df_dst_ratio)

Phase 4 dst_ratio Statistics:
count    228.000000
mean       0.330142
std        0.111354
min        0.000015
25%        0.384122
50%        0.385252
75%        0.385690
max        0.386206
Name: dst_ratio, dtype: float64


In [161]:
df_phase_4_protocols = df_phase_4["proto"].value_counts()
print("Phase 4 Protocol Distribution:")
print(df_phase_4_protocols)

Phase 4 Protocol Distribution:
proto
tcp    228
Name: count, dtype: int64


In [162]:
analyze_origin_destination(df_benign, "Benign")

Benign Origin Distribution:
local_orig
T    333964
Name: count, dtype: int64
Benign Destination Distribution:
local_resp
T    273029
F     60935
Name: count, dtype: int64


In [163]:
analyze_origin_destination(df, "All Phases")

All Phases Origin Distribution:
local_orig
T    352749
Name: count, dtype: int64
All Phases Destination Distribution:
local_resp
T    291814
F     60935
Name: count, dtype: int64


In [164]:
phase4_ips = set(df_phase_4["src_ip"]).union(set(df_phase_4["dst_ip"]))
print("Unique IPs in Phase 4:", len(phase4_ips))
print(phase4_ips)

Unique IPs in Phase 4: 7
{'10.229.2.25', '192.168.104.155', '10.229.1.118', '172.21.129.224', '10.229.0.4', '172.21.128.119', '10.229.2.216'}


In [165]:
df_phase_4[df_phase_4["dst_ip"] == "10.35.35.206"]
# {'172.17.130.196', '172.17.130.37', '192.168.255.254', '10.35.35.206', '192.168.128.4', '151.101.14.109', '192.168.130.77'}

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,start_time_dt,end_time_dt,phase,unique_sources,fanin_rate,unique_targets,fanout_rate,dst_ratio,unique_ports,connection_count


In [166]:
df_phase_4[df_phase_4["src_ip"] == "10.35.35.206"]

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,start_time_dt,end_time_dt,phase,unique_sources,fanin_rate,unique_targets,fanout_rate,dst_ratio,unique_ports,connection_count


In [167]:
def get_phase_ips(df, phase):
    print(f"=== Analyzing Phase {phase} IPs ===")
    phase_ips = set(df["src_ip"].unique()) | set(df["dst_ip"].unique())
    phase_src_ips = set(df["src_ip"].unique())
    phase_dst_ips = set(df["dst_ip"].unique())

    print(f"Unique IPs in Phase {phase}:", phase_ips)
    print(f"Unique Source IPs in Phase {phase}:", phase_src_ips)
    print(f"Unique Destination IPs in Phase {phase}:", phase_dst_ips)
    print()

    return phase_ips, phase_src_ips, phase_dst_ips

In [168]:
phase1_ips, phase1_src_ips, phase1_dst_ips = get_phase_ips(df_phase_1, 1)
phase2_ips, phase2_src_ips, phase2_dst_ips = get_phase_ips(df_phase_2, 2)
phase3_ips, phase3_src_ips, phase3_dst_ips = get_phase_ips(df_phase_3, 3)
phase4_ips, phase4_src_ips, phase4_dst_ips = get_phase_ips(df_phase_4, 4)

=== Analyzing Phase 1 IPs ===
Unique IPs in Phase 1: {'10.229.255.254', '10.229.2.216'}
Unique Source IPs in Phase 1: {'10.229.255.254'}
Unique Destination IPs in Phase 1: {'10.229.2.216'}

=== Analyzing Phase 2 IPs ===
Unique IPs in Phase 2: {'192.168.104.91', '192.168.104.38', '192.168.104.37', '192.168.104.100', '192.168.104.208', '192.168.104.129', '192.168.104.62', '192.168.104.76', '192.168.104.5', '192.168.104.121', '192.168.104.110', '192.168.104.227', '192.168.104.9', '192.168.104.200', '192.168.104.15', '192.168.104.165', '192.168.104.198', '192.168.104.88', '192.168.104.201', '192.168.104.53', '192.168.104.237', '192.168.104.169', '192.168.104.25', '192.168.104.177', '192.168.104.113', '192.168.104.35', '192.168.104.60', '192.168.104.61', '192.168.104.193', '192.168.104.247', '192.168.104.161', '192.168.104.182', '192.168.104.27', '192.168.104.210', '172.21.128.2', '192.168.104.43', '10.229.255.254', '192.168.104.134', '192.168.104.255', '192.168.104.162', '172.21.128.54', '

In [169]:
print("IPs in Phase 4:", phase4_ips)
for ip in phase4_ips:
    print(f"\nAttacker IP: {ip}")
    print()
    df_attacker_ip = df[df["src_ip"] == ip]
    df_attacker_ip_labels = df_attacker_ip["label"].value_counts()
    print("Label Distribution for this IP as a Source:")
    print(df_attacker_ip_labels)
    print()

    df_attacker_ip = df[df["dst_ip"] == ip]
    df_attacker_ip_labels = df_attacker_ip["label"].value_counts()
    print("Label Distribution for this IP as a Destination:")
    print(df_attacker_ip_labels)

IPs in Phase 4: {'10.229.2.25', '192.168.104.155', '10.229.1.118', '172.21.129.224', '10.229.0.4', '172.21.128.119', '10.229.2.216'}

Attacker IP: 10.229.2.25

Label Distribution for this IP as a Source:
label
mail                    1004
broken flow - benign      10
benign                     1
Name: count, dtype: int64



Label Distribution for this IP as a Destination:
label
mail                    793
broken flow - benign    130
online_cracking           3
benign                    1
Name: count, dtype: int64

Attacker IP: 192.168.104.155

Label Distribution for this IP as a Source:
label
monitoring                           339
benign DNS                           236
NTP                                  177
benign                               129
browsing/update                       80
broken flow - benign                  16
bootp                                  8
online_cracking                        1
update/command on unassigned port      1
Name: count, dtype: int64

Label Distribution for this IP as a Destination:
label
HTTPS                   2654
HTTP(S) intra           2191
benign                   861
service_scan             859
HTTP                     424
dirb_scan                 90
wpscan                    84
broken flow - benign      62
online_cracking           40
SSH           

In [170]:
all_phase_ips = phase1_ips | phase2_ips | phase3_ips | phase4_ips
print(len(all_phase_ips), "unique IPs across all phases.")

for i, ip in enumerate(all_phase_ips):
    in_phases = []
    if ip in phase1_ips:
        in_phases.append("Phase 1")
    if ip in phase2_ips:
        in_phases.append("Phase 2")
    if ip in phase3_ips:
        in_phases.append("Phase 3")
    if ip in phase4_ips:
        in_phases.append("Phase 4")
    
    if len(in_phases) > 1:
        print(f"IP {ip} appears in: {', '.join(in_phases)}")

268 unique IPs across all phases.
IP 10.229.255.254 appears in: Phase 1, Phase 2
IP 192.168.104.155 appears in: Phase 2, Phase 3, Phase 4
IP 172.21.129.224 appears in: Phase 2, Phase 4
IP 172.21.128.119 appears in: Phase 2, Phase 3, Phase 4
IP 10.229.2.216 appears in: Phase 1, Phase 2, Phase 4
